# Agent Gauntlet — GRPO Training Notebook

> **88% of enterprise AI agents fail when moved from demo to production.**
> This notebook trains an LLM to survive real production failure conditions.

**Runtime:** T4 GPU | **Model:** Qwen/Qwen3-1.7B (4-bit QLoRA) | **Time:** ~45 min

In [ ]:
# ── Cell 1: Clone repo ──────────────────────────────────────────────────────
import os, subprocess, sys, zipfile, glob

REPO_DIR   = '/content/agent-gauntlet'
GITHUB_URL = 'https://github.com/LakkuAmulya-2/agent-gauntlet.git'

def _try_clone(url, dest):
    r = subprocess.run(['git','clone','--depth','1', url, dest], capture_output=True, text=True)
    if r.returncode != 0:
        print(f'  clone failed: {r.stderr.strip()[:120]}')
        return False
    return True

def _try_zip(dest):
    zips = glob.glob('/content/agent-gauntlet*.zip')
    if not zips: return False
    with zipfile.ZipFile(zips[0]) as zf:
        top = zf.namelist()[0].split('/')[0]
        zf.extractall('/content/')
    extracted = f'/content/{top}'
    if os.path.isdir(extracted) and extracted != dest:
        os.rename(extracted, dest)
    return os.path.exists(dest)

if not os.path.exists(REPO_DIR):
    print(f'Cloning from GitHub...')
    if not _try_clone(GITHUB_URL, REPO_DIR):
        print('Trying uploaded zip...')
        if not _try_zip(REPO_DIR):
            raise SystemExit('Upload agent-gauntlet.zip via Files panel, then re-run.')
    print(f'Repo ready at {REPO_DIR}')
else:
    print(f'Repo already present')

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path: sys.path.insert(0, REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
# ── Cell 2: Config ──────────────────────────────────────────────────────────
HF_USERNAME   = 'amulyalakku'       # your HuggingFace username
WANDB_PROJECT = None                # set to 'agent-gauntlet' to enable wandb
DIFFICULTY    = 'easy'              # easy | medium | hard | expert
DATASET_SIZE  = 200                 # prompts per run (keep small for T4)
MODEL_ID      = 'Qwen/Qwen3-1.7B'
MAX_SEQ_LENGTH = 2048
print('Config ready. DIFFICULTY:', DIFFICULTY)

In [ ]:
# ── Cell 3: Install deps ─────────────────────────────────────────────────────
# Run this cell, then RESTART RUNTIME, then continue from Cell 4
import subprocess, sys, os

# Unsloth first
subprocess.run([sys.executable,'-m','pip','install','-q',
    'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'], check=False)
# Core deps
subprocess.run([sys.executable,'-m','pip','install','-q',
    'trl>=1.0.0','openenv-core>=0.2.1','datasets>=2.18.0',
    'wandb','matplotlib','numpy'], check=False)
# Agent Gauntlet package
repo_dir = '/content/agent-gauntlet'
if os.path.exists(os.path.join(repo_dir,'pyproject.toml')):
    subprocess.run([sys.executable,'-m','pip','install','-q','-e',repo_dir,'--no-deps'], check=False)

print('\n' + '='*60)
print('DONE. Now do: Runtime → Restart runtime')
print('Then run cells 1,2,4,5,6... (skip cell 3)')
print('='*60)

In [ ]:
# ── Cell 4: GPU check ───────────────────────────────────────────────────────
import torch
print(f'GPU:  {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'CUDA: {torch.version.cuda}')
import transformers
print(f'transformers: {transformers.__version__}')
import trl
print(f'trl: {trl.__version__}')

In [ ]:
# ── Cell 5: Connect environment ──────────────────────────────────────────────
import os
os.environ['GAUNTLET_DIRECT'] = '1'
os.environ['GAUNTLET_DIFFICULTY'] = DIFFICULTY

from agent_gauntlet.runtime.environment import AgentGauntletEnvironment
from agent_gauntlet.models import AgentAction, ActionType

env = AgentGauntletEnvironment(default_difficulty=DIFFICULTY)
obs = env.reset(difficulty=DIFFICULTY)
print('Environment connected!')
print(f'Task:    {obs.task_description[:80]}...')
print(f'Domain:  {obs.task_domain} | Difficulty: {obs.difficulty}')
print(f'Tools:   {obs.available_tools}')
print(f'Budget:  {obs.api_calls_budget} API calls | Steps: {obs.max_steps}')
print(f'Policies:{obs.active_policies}')

In [ ]:
# ── Cell 6: Random baseline ──────────────────────────────────────────────────
import random

def run_baseline(n_episodes=10, difficulty='easy'):
    rewards, completed, detected = [], [], []
    env = AgentGauntletEnvironment(default_difficulty=difficulty)
    for _ in range(n_episodes):
        obs = env.reset(difficulty=difficulty)
        ep_reward = 0.0
        while not obs.is_done:
            action = AgentAction(
                action_type=ActionType.CALL_TOOL.value,
                tool_name=random.choice(obs.available_tools),
                reasoning='Random baseline'
            )
            obs = env.step(action)
            ep_reward += getattr(obs, '_reward', 0.0)
        rewards.append(ep_reward)
        completed.append(1 if obs.termination_reason == 'task_completed' else 0)
        meta = obs.metadata or {}
        detected.append(meta.get('failures_detected_correctly', 0))
    n = len(rewards)
    return {
        'avg_reward': sum(rewards)/n,
        'task_completion_rate': sum(completed)/n,
        'avg_failures_detected': sum(detected)/n,
        'all_rewards': rewards,
    }

print(f'Running random baseline (10 episodes, {DIFFICULTY})...')
baseline = run_baseline(n_episodes=10, difficulty=DIFFICULTY)
print(f'Avg reward:       {baseline["avg_reward"]:.4f}')
print(f'Task completion:  {baseline["task_completion_rate"]:.1%}')
print(f'Failures detected:{baseline["avg_failures_detected"]:.1f}')

In [ ]:
# ── Cell 7: System prompt ────────────────────────────────────────────────────
import sys
sys.path.insert(0, '/content/agent-gauntlet')
try:
    from train_grpo import SYSTEM_PROMPT
    print(f'SYSTEM_PROMPT loaded: {len(SYSTEM_PROMPT)} chars')
except ImportError:
    SYSTEM_PROMPT = """You are a production AI agent. Complete enterprise tasks while handling failures.
Respond with JSON: {"action_type": "call_tool", "tool_name": "<tool>", "reasoning": "<why>"}
Failure types: api_500, rate_limit_429, auth_401, timeout, cascading, security_breach, compliance_violation
After failures: {"action_type": "detect_failure", "failure_detected": "<type>", "reasoning": "<evidence>"}
Then recover: {"action_type": "recover", "recovery_strategy": "<strategy>", "reasoning": "<plan>"}"""
    print('Using fallback SYSTEM_PROMPT')

In [ ]:
# ── Cell 8: Dataset ──────────────────────────────────────────────────────────
from datasets import Dataset

DOMAINS = ['data_pipeline','api_workflow','report_generation','system_config',
           'multi_agent_coordination','code_review','incident_response',
           'personal_assistant','large_scale_migration']
CONTEXTS = [
    'A new enterprise task has been assigned.',
    'This is a critical production workflow.',
    'Handle carefully — failures have business impact.',
    'Security and compliance policies are active.',
    'Long-running task — checkpoint state regularly.',
]

prompts = []
for i in range(DATASET_SIZE):
    domain  = DOMAINS[i % len(DOMAINS)]
    context = CONTEXTS[i % len(CONTEXTS)]
    prompts.append([{
        'role': 'system', 'content': SYSTEM_PROMPT
    }, {
        'role': 'user',
        'content': (
            f'You are in a {DIFFICULTY} difficulty production environment. '
            f'{context} Domain: {domain.replace("_"," ")}. '
            'Complete the task step by step using JSON actions. '
            'Detect and recover from any failures you encounter.'
        )
    }])

dataset = Dataset.from_dict({'prompt': prompts})
print(f'Dataset: {len(dataset)} samples')

In [ ]:
# ── Cell 9: Load model + build trainer ──────────────────────────────────────
#
# NOTE: We do NOT use environment_factory — it requires transformers>=5.2.0
# which conflicts with unsloth on Colab T4.
# Instead: environment runs INLINE inside the reward function.
# This works with any transformers version and is fully equivalent.
#
import unsloth  # must be first import
from trl import GRPOConfig, GRPOTrainer
from transformers import AutoTokenizer
from datetime import datetime
import torch, json as _json, os, inspect

OUTPUT_DIR = f'outputs/gauntlet-{DIFFICULTY}-{datetime.now().strftime("%Y%m%d_%H%M%S")}'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Load model ───────────────────────────────────────────────────────────────
try:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_ID, max_seq_length=MAX_SEQ_LENGTH,
        dtype=None, load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model, r=16,
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        lora_alpha=16, lora_dropout=0, bias='none',
        use_gradient_checkpointing='unsloth',
    )
    print(f'Loaded with Unsloth (4-bit QLoRA): {MODEL_ID}')
except ImportError:
    model = MODEL_ID
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    tokenizer.pad_token = tokenizer.eos_token
    print(f'Unsloth not available: {MODEL_ID}')

# ── GPU precision ─────────────────────────────────────────────────────────────
gpu_compute = torch.cuda.get_device_capability()[0] if torch.cuda.is_available() else 0
use_bf16 = gpu_compute >= 8   # Ampere+ only (A100, A10G)
use_fp16 = not use_bf16 and torch.cuda.is_available()  # T4 uses fp16
print(f'GPU compute {gpu_compute}.x → bf16={use_bf16}, fp16={use_fp16}')

# ── Inline reward function ────────────────────────────────────────────────────
from agent_gauntlet.runtime.environment import AgentGauntletEnvironment
from agent_gauntlet.models import AgentAction

_ENV_POOL = {}

def _get_env(idx=0):
    if idx not in _ENV_POOL:
        _ENV_POOL[idx] = AgentGauntletEnvironment(default_difficulty=DIFFICULTY, seed=idx)
    return _ENV_POOL[idx]

def _parse_action(text):
    try:
        s = text.find('{')
        e = text.rfind('}') + 1
        if s == -1 or e == 0: raise ValueError
        d = _json.loads(text[s:e])
        return AgentAction(
            action_type=d.get('action_type','call_tool'),
            tool_name=d.get('tool_name'),
            tool_args=d.get('tool_args',{}),
            reasoning=d.get('reasoning',''),
            failure_detected=d.get('failure_detected'),
            recovery_strategy=d.get('recovery_strategy'),
            task_result=d.get('task_result'),
            injection_refused=d.get('injection_refused', False),
            compliance_check_result=d.get('compliance_check_result'),
            compliance_alternative=d.get('compliance_alternative'),
            decision_documented=d.get('decision_documented'),
            diagnostic_trace=d.get('diagnostic_trace'),
            transparency_decision=d.get('transparency_decision'),
            checkpoint_data=d.get('checkpoint_data'),
            drift_detected=d.get('drift_detected'),
        )
    except Exception:
        return AgentAction(action_type='call_tool', reasoning=text[:80])

def gauntlet_reward(completions, prompts=None, **kwargs):
    """Run Agent Gauntlet environment inline — no environment_factory needed."""
    rewards = []
    for i, completion in enumerate(completions):
        text = ' '.join(m.get('content','') for m in completion) if isinstance(completion, list) else str(completion)
        try:
            env = _get_env(i % 4)
            obs = env.reset(difficulty=DIFFICULTY)
            action = _parse_action(text)
            obs = env.step(action)
            rewards.append(float(getattr(obs, '_reward', 0.0)))
        except Exception:
            rewards.append(-0.1)
    return rewards

# ── GRPOConfig ────────────────────────────────────────────────────────────────
_grpo_params = set(inspect.signature(GRPOConfig).parameters.keys())

grpo_kwargs = dict(
    num_train_epochs=1,
    learning_rate=5e-6,
    gradient_accumulation_steps=16,
    per_device_train_batch_size=1,
    num_generations=2,
    max_completion_length=256,
    bf16=use_bf16,
    fp16=use_fp16,
    output_dir=OUTPUT_DIR,
    logging_steps=1,
    save_steps=25,
    report_to='wandb' if WANDB_PROJECT else 'none',
    run_name=f'gauntlet-{DIFFICULTY}-{datetime.now().strftime("%Y%m%d_%H%M%S")}',
    log_completions=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
)
if 'max_prompt_length' in _grpo_params:
    grpo_kwargs['max_prompt_length'] = 512
if 'chat_template_kwargs' in _grpo_params:
    grpo_kwargs['chat_template_kwargs'] = {'enable_thinking': False}

grpo_config = GRPOConfig(**grpo_kwargs)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[gauntlet_reward],
    train_dataset=dataset,
    args=grpo_config,
)

print(f'Trainer ready!')
print(f'Model:  {MODEL_ID}')
print(f'Output: {OUTPUT_DIR}')

In [ ]:
# ── Cell 10: Train ───────────────────────────────────────────────────────────
import os
os.environ['TRL_EXPERIMENTAL_SILENCE'] = '1'  # silence environment_factory warning
trainer_stats = trainer.train()
print(f'Training complete in {trainer_stats.metrics["train_runtime"]:.0f}s')

In [ ]:
# ── Cell 11: Plot reward curves ──────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

log_history = trainer.state.log_history
steps   = [x['step']   for x in log_history if 'reward' in x]
rewards = [x['reward'] for x in log_history if 'reward' in x]

if not rewards:
    print('No reward history yet')
else:
    window = max(1, min(10, len(rewards)//4))
    smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')

    plt.figure(figsize=(10,5))
    plt.plot(steps, rewards, alpha=0.3, color='steelblue', label='Per-step reward')
    plt.plot(steps[window-1:], smoothed, color='steelblue', lw=2, label=f'Rolling avg (w={window})')
    plt.axhline(y=baseline['avg_reward'], color='red', linestyle='--',
                label=f'Random baseline ({baseline["avg_reward"]:.3f})')
    plt.xlabel('Training step')
    plt.ylabel('Reward')
    plt.title('Agent Gauntlet — GRPO Training Reward')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('reward_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: reward_curves.png')
    final = float(np.mean(rewards[-20:])) if len(rewards)>=20 else float(np.mean(rewards))
    print(f'Baseline: {baseline["avg_reward"]:.4f} → Trained: {final:.4f}')

In [ ]:
# ── Cell 12: Save model ──────────────────────────────────────────────────────
# Do NOT upcast 4-bit to 16-bit naively — use trainer.save_model()
trainer.save_model(OUTPUT_DIR)
print(f'Model saved to {OUTPUT_DIR}')